> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
from pathlib import Path
import os
import subprocess

ROOT = Path("/content/TFM_Rendimiento_cultivos")

# Si el notebook se abre directamente desde GitHub en Colab,
# descargamos automáticamente el repositorio.
if not ROOT.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ritaridh/tfm-crop-yield-prediction.git",
            str(ROOT),
        ],
        check=True,
    )

os.chdir(ROOT)

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS

Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)

print("Repositorio:", ROOT)


# Exploración de EuroCrops España 2023

## Objetivo

Comprobar la estructura de los archivos provinciales de EuroCrops España 2023 y generar posteriormente un inventario de parcelas, cultivos y superficies por provincia.

## Fuente

EuroCrops v11, España 2023.

## Alcance de esta primera prueba

En esta fase se analiza únicamente el archivo DBF de Alicante para comprobar:

- lectura correcta del archivo;
- nombres y tipos de columnas;
- número de registros;
- variables de cultivo;
- superficies;
- valores ausentes.

#Montar Google Drive

#Definir Rutas

In [ ]:
from pathlib import Path

# Carpeta principal del proyecto en Google Drive
BASE_DIR = Path(
    "/content/TFM_Rendimiento_cultivos"
)

# Carpeta que contiene los archivos DBF provinciales
DBF_DIR = BASE_DIR / "01_Datos_originales" / "EuroCrops" / "ES_2023_DBF"

print("Carpeta principal:", BASE_DIR)
print("Carpeta DBF:", DBF_DIR)
print("¿Existe la carpeta DBF?:", DBF_DIR.exists())

In [ ]:
import os
import pandas as pd
from pathlib import Path

# ============================================================
# RUTAS PERMANENTES DEL TFM EN GOOGLE DRIVE
# ============================================================

RUTA_TFM = "/content/TFM_Rendimiento_cultivos"

RUTA_ORIGINALES = os.path.join(
    RUTA_TFM,
    "01_Datos_originales"
)

RUTA_EUROCROPS = os.path.join(
    RUTA_ORIGINALES,
    "EuroCrops"
)

RUTA_MAPA = os.path.join(
    RUTA_ORIGINALES,
    "MAPA"
)

RUTA_PROCESADOS = os.path.join(
    RUTA_TFM,
    "02_Datos_procesados"
)

RUTA_RESULTADOS = os.path.join(
    RUTA_TFM,
    "05_Resultados"
)

print("RUTA TFM:", RUTA_TFM)
print("EuroCrops:", RUTA_EUROCROPS)
print("MAPA:", RUTA_MAPA)
print("Procesados:", RUTA_PROCESADOS)
print("Resultados:", RUTA_RESULTADOS)

print("\nComprobación:")
print("TFM existe:", os.path.exists(RUTA_TFM))
print("EuroCrops existe:", os.path.exists(RUTA_EUROCROPS))
print("MAPA existe:", os.path.exists(RUTA_MAPA))
print("Procesados existe:", os.path.exists(RUTA_PROCESADOS))
print("Resultados existe:", os.path.exists(RUTA_RESULTADOS))

#Comprobar que DBF está visible

In [ ]:
dbf_files = sorted(DBF_DIR.glob("*.dbf"))

print("Número de archivos DBF encontrados:", len(dbf_files))

for file in dbf_files:
    print(file.name)

In [ ]:
from pathlib import Path

DBF_DIR = Path(
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/EuroCrops/ES_2023_DBF"
)

print("DBF_DIR:", DBF_DIR)
print("Existe:", DBF_DIR.exists())

#Instalar libreria de lectura DBF

In [ ]:
!pip install -q dbfread

#Leer Alicante

In [ ]:
import os

print("Buscando es_a_2023.dbf...\n")

for raiz, carpetas, archivos in os.walk(RUTA_EUROCROPS):
    if "es_a_2023.dbf" in archivos:
        print("ENCONTRADO:")
        print(os.path.join(raiz, "es_a_2023.dbf"))

In [ ]:
from dbfread import DBF
import pandas as pd

alicante_file = DBF_DIR / "es_a_2023.dbf"

table = DBF(
    alicante_file,
    encoding="utf-8",
    load=True
)

df_alicante = pd.DataFrame(iter(table))

print("Archivo leído:", alicante_file.name)
print("Número de filas:", len(df_alicante))
print("Número de columnas:", len(df_alicante.columns))

In [ ]:
# Mostras columnas

print("Columnas disponibles:")

for index, column in enumerate(df_alicante.columns, start=1):
    print(f"{index:02d}. {column}")

In [ ]:
# Primeras filas

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

df_alicante.head(10)

In [ ]:
# Revisar tipo de datos

df_alicante.info()

In [ ]:
# Comprobar año y província

print("Años disponibles:")
print(df_alicante["exp_ano"].value_counts(dropna=False))

print("\nCódigos de provincia:")
print(df_alicante["provincia"].value_counts(dropna=False))

In [ ]:
# Comprobar cultivos

print("Número de categorías HCAT distintas:")
print(df_alicante["EC_hcat_n"].nunique(dropna=True))

print("\n20 cultivos con mayor número de parcelas:")
print(
    df_alicante["EC_hcat_n"]
    .value_counts(dropna=False)
    .head(20)
)

In [ ]:
#Comprobar superfície

surface_columns = ["dn_surface", "par_supcul"]

for column in surface_columns:
    print(f"\nResumen de {column}:")
    print(pd.to_numeric(df_alicante[column], errors="coerce").describe())

#Resumen de prueba

In [ ]:
RESULTS_DIR = BASE_DIR / "05_Resultados" / "EuroCrops"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de resultados:", RESULTS_DIR)

In [ ]:
summary_alicante = pd.DataFrame({
    "provincia_archivo": ["es_a_2023"],
    "numero_parcelas": [len(df_alicante)],
    "numero_columnas": [len(df_alicante.columns)],
    "numero_cultivos_hcat": [df_alicante["EC_hcat_n"].nunique(dropna=True)],
    "ano": [df_alicante["exp_ano"].dropna().unique().tolist()],
})

summary_path = RESULTS_DIR / "resumen_prueba_alicante.csv"
summary_alicante.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(summary_path)

summary_alicante

#Comprobar valores vacíod en las variables HCAT

In [ ]:
# Comprobar valores NULL, cadenas vacías y espacios
hcat_columns = ["EC_trans_n", "EC_hcat_n", "EC_hcat_c"]

for column in hcat_columns:
    nulls = df_alicante[column].isna().sum()

    # Convertimos temporalmente a texto para detectar cadenas vacías
    blanks = (
        df_alicante[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{column}")
    print(f"  Valores NULL: {nulls}")
    print(f"  Valores vacíos: {blanks}")

In [ ]:
# Mostrar los registros cuya categoría HCAT está vacía
mask_blank_hcat = (
    df_alicante["EC_hcat_n"]
    .astype(str)
    .str.strip()
    .eq("")
)

df_alicante.loc[
    mask_blank_hcat,
    [
        "par_produc",
        "EC_trans_n",
        "EC_hcat_n",
        "EC_hcat_c",
        "par_supcul"
    ]
].head(20)

In [ ]:
# Frecuencia de códigos originales entre los registros sin HCAT_name
(
    df_alicante.loc[mask_blank_hcat, "par_produc"]
    .value_counts(dropna=False)
    .head(20)
)

In [ ]:
# Ver traducción y código HCAT asociados
(
    df_alicante.loc[
        mask_blank_hcat,
        ["par_produc", "EC_trans_n", "EC_hcat_c"]
    ]
    .drop_duplicates()
    .head(50)
)

# Comprobar que dn_surface y par_supcul son realmente equivalentes

In [ ]:
import numpy as np

# Diferencia entre ambas variables
df_alicante["diferencia_superficie"] = (
    df_alicante["dn_surface"]
    - df_alicante["par_supcul"]
)

print("Resumen de la diferencia:")
print(df_alicante["diferencia_superficie"].describe())

print("\nRegistros con exactamente la misma superficie:")
print(
    np.isclose(
        df_alicante["dn_surface"],
        df_alicante["par_supcul"],
        rtol=0,
        atol=0.01
    ).sum()
)

print("\nNúmero total de registros:")
print(len(df_alicante))

In [ ]:
# Registros con las mayores diferencias entre ambas superficies
df_alicante[
    [
        "dn_oid",
        "dn_surface",
        "par_supcul",
        "diferencia_superficie",
        "EC_hcat_n"
    ]
].assign(
    diferencia_abs=lambda x: x["diferencia_superficie"].abs()
).sort_values(
    "diferencia_abs",
    ascending=False
).head(20)

#comprobar valores imposibles o problemáticos de superfície

In [ ]:
for column in ["dn_surface", "par_supcul"]:
    print(f"\n{column}")
    print("Valores = 0:", (df_alicante[column] == 0).sum())
    print("Valores < 0:", (df_alicante[column] < 0).sum())
    print("Valores ausentes:", df_alicante[column].isna().sum())

In [ ]:
# Ver parcelas con par_supcul igual a 0
df_alicante.loc[
    df_alicante["par_supcul"] == 0,
    [
        "dn_oid",
        "dn_surface",
        "par_supcul",
        "par_produc",
        "EC_trans_n",
        "EC_hcat_n"
    ]
].head(20)

#Obtener ranking más útil de cultivos de Alicante

In [ ]:
resumen_cultivos_alicante = (
    df_alicante
    .assign(
        EC_hcat_n_limpio=
            df_alicante["EC_hcat_n"]
            .astype(str)
            .str.strip()
            .replace("", "SIN_CLASIFICAR")
    )
    .groupby("EC_hcat_n_limpio", dropna=False)
    .agg(
        numero_parcelas=("dn_oid", "count"),
        superficie_total=("par_supcul", "sum"),
        superficie_media=("par_supcul", "mean")
    )
    .reset_index()
    .sort_values("superficie_total", ascending=False)
)

resumen_cultivos_alicante.head(30)

# Comprobar distribución territorial

In [ ]:
resumen_municipios_cultivo = (
    df_alicante
    .assign(
        EC_hcat_n_limpio=
            df_alicante["EC_hcat_n"]
            .astype(str)
            .str.strip()
            .replace("", "SIN_CLASIFICAR")
    )
    .groupby("EC_hcat_n_limpio")
    .agg(
        numero_parcelas=("dn_oid", "count"),
        numero_municipios=("municipio", "nunique"),
        superficie_total=("par_supcul", "sum")
    )
    .reset_index()
    .sort_values(
        ["numero_municipios", "superficie_total"],
        ascending=[False, False]
    )
)

resumen_municipios_cultivo.head(30)

#Guardar resumenes

In [ ]:
resumen_cultivos_alicante.to_csv(
    RESULTS_DIR / "cultivos_alicante_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_municipios_cultivo.to_csv(
    RESULTS_DIR / "cultivos_municipios_alicante_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos guardados correctamente.")

#Ultima comprobación para Alicante: verificar que no existen duplicados en el identificador que aparentemente debería identificar cada recinto.

In [ ]:
for col in df_alicante.columns:
    print(repr(col))

In [ ]:
id_columns = [
    "dn_oid",
    "ld_recinto",
    "exp_num"
]

for column in id_columns:
    print(f"\nVariable: {column}")
    print("Número de registros:", len(df_alicante))
    print("Valores únicos:", df_alicante[column].nunique(dropna=False))
    print(
        "Registros duplicados:",
        df_alicante[column].duplicated(keep=False).sum()
    )

In [ ]:
duplicados_ld_recinto = df_alicante[
    df_alicante["ld_recinto"].duplicated(keep=False)
].sort_values("ld_recinto")

duplicados_ld_recinto[
    [
        "dn_oid",
        "ld_recinto",
        "provincia",
        "municipio",
        "poligono",
        "parcela",
        "recinto",
        "par_produc",
        "EC_hcat_n",
        "par_supcul"
    ]
].head(30)

# Todos los archivos de España

In [ ]:
dbf_files = sorted(DBF_DIR.glob("*.dbf"))

print("Número de archivos DBF encontrados:", len(dbf_files))

for file in dbf_files:
    print(file.name)

In [ ]:
# Comprobar estructura - ver si todos tienen la misma estructura

from dbfread import DBF
import pandas as pd

estructura_archivos = []

for file in dbf_files:
    table = DBF(
        file,
        encoding="utf-8",
        load=False
    )

    columnas = [field.name for field in table.fields]

    estructura_archivos.append({
        "archivo": file.name,
        "numero_columnas": len(columnas),
        "columnas": columnas
    })

estructura_df = pd.DataFrame(estructura_archivos)

estructura_df[["archivo", "numero_columnas"]]

In [ ]:
# Comprobar estructura

estructura_df["numero_columnas"].value_counts().sort_index()

In [ ]:
columnas_referencia = estructura_df.iloc[0]["columnas"]

archivos_diferentes = []

for _, row in estructura_df.iterrows():
    if row["columnas"] != columnas_referencia:
        archivos_diferentes.append(row["archivo"])

print("Archivos con estructura diferente:")
print(archivos_diferentes)

print("\nNúmero de archivos diferentes:")
print(len(archivos_diferentes))

# Crear inventario básico provincia por provincia

In [ ]:
inventario_provincias = []

for i, file in enumerate(dbf_files, start=1):

    print(f"Procesando {i:02d}/50: {file.name}")

    table = DBF(
        file,
        encoding="utf-8",
        load=True
    )

    df = pd.DataFrame(iter(table))

    # Limpiar posibles espacios en nombres de columnas
    df.columns = df.columns.str.strip()

    # Categorías HCAT vacías
    hcat_vacio = (
        df["EC_hcat_n"]
        .astype(str)
        .str.strip()
        .eq("")
    )

    # Valores de provincia y año
    provincias = df["provincia"].dropna().unique().tolist()
    anos = df["exp_ano"].dropna().unique().tolist()

    inventario_provincias.append({
        "archivo": file.name,
        "codigo_provincia": provincias,
        "ano": anos,
        "numero_registros": len(df),
        "numero_municipios": df["municipio"].nunique(dropna=True),
        "numero_categorias_hcat": df.loc[~hcat_vacio, "EC_hcat_n"].nunique(),
        "registros_sin_clasificar": int(hcat_vacio.sum()),
        "porcentaje_sin_clasificar": round(hcat_vacio.mean() * 100, 2),
        "superficie_total_par_supcul": pd.to_numeric(
            df["par_supcul"],
            errors="coerce"
        ).sum(),
        "duplicados_dn_oid": int(
            df["dn_oid"].duplicated(keep=False).sum()
        )
    })

inventario_provincias_df = pd.DataFrame(inventario_provincias)

inventario_provincias_df

In [ ]:
# Comprobar controles basicos

print("Número de provincias procesadas:")
print(len(inventario_provincias_df))

print("\nAños encontrados:")
print(inventario_provincias_df["ano"].value_counts())

print("\nDuplicados dn_oid por provincia:")
print(
    inventario_provincias_df[
        inventario_provincias_df["duplicados_dn_oid"] > 0
    ][["archivo", "duplicados_dn_oid"]]
)

print("\nTotal de registros de España:")
print(inventario_provincias_df["numero_registros"].sum())

In [ ]:
#Provincias con más registros

inventario_provincias_df[
    [
        "archivo",
        "codigo_provincia",
        "numero_registros",
        "numero_municipios",
        "numero_categorias_hcat",
        "registros_sin_clasificar",
        "porcentaje_sin_clasificar"
    ]
].sort_values(
    "numero_registros",
    ascending=False
).head(15)

In [ ]:
# Guardar inventario

inventario_path = (
    RESULTS_DIR /
    "inventario_provincias_eurocrops_es_2023.csv"
)

inventario_provincias_df.to_csv(
    inventario_path,
    index=False,
    encoding="utf-8-sig"
)

print("Inventario guardado en:")
print(inventario_path)

## 3. Inventario nacional de cultivos

Se procesan individualmente los 50 archivos provinciales de EuroCrops España 2023.

En esta fase no se eliminan provincias, cultivos ni registros. Los valores sin clasificación HCAT se conservan y se identifican únicamente en las tablas derivadas como `SIN_CLASIFICAR`.

El análisis se realiza mediante agregación provincial para evitar cargar simultáneamente los más de 18 millones de registros del conjunto nacional.

In [ ]:
# Confirmar el tipo de par_supcul en Barcelona
# Comprobar cómo se lee par_supcul en Barcelona

file_barcelona = DBF_DIR / "es_b_2023.dbf"

table_bcn = DBF(
    file_barcelona,
    encoding="utf-8",
    load=True
)

df_bcn = pd.DataFrame(iter(table_bcn))
df_bcn.columns = df_bcn.columns.str.strip()

print("Tipo de par_supcul:")
print(df_bcn["par_supcul"].dtype)

print("\nPrimeros 20 valores:")
print(df_bcn["par_supcul"].head(20))

print("\nTipos Python encontrados:")
print(df_bcn["par_supcul"].map(type).value_counts())

In [ ]:
# Convertir par_supcul a numérico (en Barcelona)

df["par_supcul_num"] = pd.to_numeric(
    df["par_supcul"],
    errors="coerce"
)

In [ ]:
# Controlar que la conversión no genera valores perdidos

from dbfread import DBF
import pandas as pd

resumen_provincia_cultivo = []

# Registro de posibles problemas durante la conversión
control_conversion_superficie = []

for i, file in enumerate(dbf_files, start=1):

    print(f"Procesando {i:02d}/{len(dbf_files)}: {file.name}")

    # -------------------------------------------------------
    # 1. Leer una provincia
    # -------------------------------------------------------

    table = DBF(
        file,
        encoding="utf-8",
        load=True
    )

    df = pd.DataFrame(iter(table))
    df.columns = df.columns.str.strip()

    # -------------------------------------------------------
    # 2. Crear categoría auxiliar de cultivo
    #    NO modificamos EC_hcat_n original
    # -------------------------------------------------------

    df["cultivo_resumen"] = (
        df["EC_hcat_n"]
        .astype(str)
        .str.strip()
        .replace("", "SIN_CLASIFICAR")
    )

    # -------------------------------------------------------
    # 3. Convertir superficie explícitamente a numérico
    #    SIN modificar par_supcul original
    # -------------------------------------------------------

    valores_no_nulos_original = df["par_supcul"].notna().sum()

    df["par_supcul_num"] = pd.to_numeric(
        df["par_supcul"],
        errors="coerce"
    )

    valores_no_nulos_convertidos = df["par_supcul_num"].notna().sum()

    problemas_conversion = (
        valores_no_nulos_original
        - valores_no_nulos_convertidos
    )

    control_conversion_superficie.append({
        "archivo": file.name,
        "numero_registros": len(df),
        "dtype_original_par_supcul": str(df["par_supcul"].dtype),
        "problemas_conversion_par_supcul": problemas_conversion
    })

    # -------------------------------------------------------
    # 4. Provincia y año
    # -------------------------------------------------------

    codigo_provincia = df["provincia"].iloc[0]
    ano = df["exp_ano"].iloc[0]

    # -------------------------------------------------------
    # 5. Identificador municipal compuesto
    # -------------------------------------------------------

    df["id_municipio_compuesto"] = (
        df["provincia"].astype(str)
        + "_"
        + df["municipio"].astype(str)
    )

    # -------------------------------------------------------
    # 6. Agregación provincia × cultivo
    # -------------------------------------------------------

    resumen = (
        df
        .groupby("cultivo_resumen", dropna=False)
        .agg(
            numero_registros=(
                "dn_oid",
                "count"
            ),
            numero_municipios=(
                "id_municipio_compuesto",
                "nunique"
            ),
            superficie_total_par_supcul=(
                "par_supcul_num",
                "sum"
            ),
            superficie_media_par_supcul=(
                "par_supcul_num",
                "mean"
            ),
            superficie_mediana_par_supcul=(
                "par_supcul_num",
                "median"
            )
        )
        .reset_index()
    )

    # -------------------------------------------------------
    # 7. Añadir metadatos
    # -------------------------------------------------------

    resumen["archivo"] = file.name
    resumen["codigo_provincia"] = codigo_provincia
    resumen["ano"] = ano

    resumen_provincia_cultivo.append(resumen)

    # -------------------------------------------------------
    # 8. Liberar memoria
    # -------------------------------------------------------

    del df
    del table
    del resumen

print("\nProcesamiento terminado.")

In [ ]:
control_conversion_df = pd.DataFrame(
    control_conversion_superficie
)

display(control_conversion_df)

In [ ]:
print("Número de provincias con problemas de conversión:")

print(
    (
        control_conversion_df[
            "problemas_conversion_par_supcul"
        ] > 0
    ).sum()
)

print("\nProvincias con problemas:")

display(
    control_conversion_df[
        control_conversion_df[
            "problemas_conversion_par_supcul"
        ] > 0
    ]
)

# Unir los resúmenes provinciales y construir el inventario nacional

In [ ]:
cultivos_por_provincia_df = pd.concat(
    resumen_provincia_cultivo,
    ignore_index=True
)

print("Número de filas de la tabla agregada:")
print(len(cultivos_por_provincia_df))

print("\nPrimeras 20 filas:")
display(cultivos_por_provincia_df.head(20))

In [ ]:
print("Número de provincias presentes:")
print(cultivos_por_provincia_df["codigo_provincia"].nunique())

print("\nCódigos provinciales:")

codigos_provinciales = (
    cultivos_por_provincia_df["codigo_provincia"]
    .astype(str)
    .unique()
    .tolist()
)

print(sorted(codigos_provinciales, key=lambda x: int(x)))

In [ ]:
#Numero total de categorias HCAT en España

numero_categorias_nacionales = (
    cultivos_por_provincia_df["cultivo_resumen"]
    .nunique()
)

print(
    "Número total de categorías HCAT en España:",
    numero_categorias_nacionales
)

In [ ]:
# Resuemn nacional

resumen_nacional_cultivos = (
    cultivos_por_provincia_df
    .groupby("cultivo_resumen", dropna=False)
    .agg(
        numero_registros=("numero_registros", "sum"),
        numero_provincias=("codigo_provincia", "nunique"),
        numero_municipios=("numero_municipios", "sum"),
        superficie_total_par_supcul=("superficie_total_par_supcul", "sum")
    )
    .reset_index()
)

resumen_nacional_cultivos["superficie_media_par_supcul"] = (
    resumen_nacional_cultivos["superficie_total_par_supcul"]
    / resumen_nacional_cultivos["numero_registros"]
)

In [ ]:
# TOTAL NACIONAL

total_registros_nacional = (
    resumen_nacional_cultivos["numero_registros"].sum()
)

print("Total nacional reconstruido:")
print(total_registros_nacional)

# Saber qué categorías dominan el dataset nacional y cómo están distribuidas.

In [ ]:


ranking_nacional = (
    resumen_nacional_cultivos
    .sort_values(
        by="numero_registros",
        ascending=False
    )
    .reset_index(drop=True)
)

# Mostrar los 30 primeros
ranking_nacional[
    [
        "cultivo_resumen",
        "numero_registros",
        "numero_provincias",
        "superficie_total_par_supcul",
        "superficie_media_par_supcul"
    ]
].head(30)

# distribución provincial de los cultivos candidatos

In [ ]:
cultivos_candidatos = [
    "barley",
    "common_soft_wheat",
    "olive_plantations",
    "vineyards_wine_vine_rebland_grapes",
    "almond",
    "sunflower",
    "oats",
    "grain_maize_corn_popcorn"
]

distribucion_candidatos = (
    cultivos_por_provincia_df[
        cultivos_por_provincia_df["cultivo_resumen"].isin(cultivos_candidatos)
    ]
    [
        [
            "codigo_provincia",
            "cultivo_resumen",
            "numero_registros",
            "superficie_total_par_supcul"
        ]
    ]
    .copy()
)

tabla_provincias_cultivos = (
    distribucion_candidatos
    .pivot_table(
        index="codigo_provincia",
        columns="cultivo_resumen",
        values="numero_registros",
        aggfunc="sum",
        fill_value=0
    )
)

tabla_provincias_cultivos

# ¿en cuántas provincias tiene una presencia realmente relevante?

In [ ]:
# Evaluar la distribución territorial de los cultivos candidatos
# SIN eliminar provincias ni parcelas

umbrales = [100, 500, 1000, 5000, 10000]

resumen_distribucion = []

for cultivo in cultivos_candidatos:

    datos = distribucion_candidatos[
        distribucion_candidatos["cultivo_resumen"] == cultivo
    ]

    fila = {
        "cultivo": cultivo,
        "parcelas_total": datos["numero_registros"].sum(),
        "provincias_con_presencia": (datos["numero_registros"] > 0).sum()
    }

    for umbral in umbrales:
        fila[f"provincias_>=_{umbral}"] = (
            datos["numero_registros"] >= umbral
        ).sum()

    resumen_distribucion.append(fila)

resumen_distribucion_df = pd.DataFrame(resumen_distribucion)

resumen_distribucion_df = (
    resumen_distribucion_df
    .sort_values("parcelas_total", ascending=False)
    .reset_index(drop=True)
)

resumen_distribucion_df

In [ ]:
# Ranking de las 15 provincias con más parcelas
# para cada cultivo candidato

for cultivo in cultivos_candidatos:

    print("\n" + "=" * 70)
    print(cultivo.upper())
    print("=" * 70)

    top = (
        distribucion_candidatos[
            distribucion_candidatos["cultivo_resumen"] == cultivo
        ]
        .sort_values("numero_registros", ascending=False)
        [
            [
                "codigo_provincia",
                "numero_registros",
                "superficie_total_par_supcul"
            ]
        ]
        .head(15)
    )

    display(top)

## Evaluación de cultivos candidatos para el modelado

La selección de los cultivos objetivo se realizará mediante criterios objetivos
relacionados con:

1. Representación en EuroCrops España 2023.
2. Distribución territorial.
3. Disponibilidad de estadísticas oficiales provinciales de rendimiento del MAPA.
4. Correspondencia entre la categoría armonizada HCAT de EuroCrops y la
   categoría estadística del MAPA.
5. Complejidad metodológica para la construcción de la variable objetivo.

En esta fase no se elimina ningún cultivo, provincia ni registro.
La matriz se utiliza exclusivamente para evaluar la viabilidad de los
cultivos candidatos.


In [ ]:
# Recuperar automaticamente las cifras de Eurocrops

cultivos_evaluacion = [
    "barley",
    "common_soft_wheat",
    "sunflower",
    "olive_plantations",
    "vineyards_wine_vine_rebland_grapes",
    "almond"
]

eurocrops_candidatos = (
    resumen_distribucion_df[
        resumen_distribucion_df["cultivo"].isin(cultivos_evaluacion)
    ]
    .copy()
)

eurocrops_candidatos

In [ ]:
# Añadir información del MAPA

informacion_mapa = pd.DataFrame({

    "cultivo": [
        "barley",
        "common_soft_wheat",
        "sunflower",
        "olive_plantations",
        "vineyards_wine_vine_rebland_grapes",
        "almond"
    ],

    "nombre_mapa": [
        "Cebada",
        "Trigo blando y semi",
        "Girasol",
        "Total olivar",
        "Total viñedo",
        "Almendro (cáscara)"
    ],

    "rendimiento_provincial_2023": [
        "Sí",
        "Sí",
        "Sí",
        "Sí",
        "Sí",
        "Sí"
    ],

    "rendimiento_secano_regadio": [
        "Sí",
        "Sí",
        "Sí",
        "Sí",
        "Sí",
        "Sí"
    ],

    "correspondencia_eurocrops_mapa": [
        "Directa",
        "Requiere integración",
        "Directa",
        "Directa",
        "Directa",
        "Directa"
    ],

    "observacion_correspondencia": [
        "EuroCrops barley ↔ MAPA Cebada resumen general",
        "EuroCrops common_soft_wheat no distingue invierno/primavera; MAPA sí",
        "EuroCrops sunflower ↔ MAPA Girasol",
        "EuroCrops olive_plantations ↔ MAPA Total olivar",
        "EuroCrops vineyards_wine_vine_rebland_grapes ↔ MAPA Total viñedo",
        "EuroCrops almond ↔ MAPA Almendro (cáscara)"
    ]
})

display(informacion_mapa)

In [ ]:
tipo_cultivo = pd.DataFrame({

    "cultivo": [
        "barley",
        "common_soft_wheat",
        "sunflower",
        "olive_plantations",
        "vineyards_wine_vine_rebland_grapes",
        "almond"
    ],

    "tipo_agronomico": [
        "Herbáceo anual",
        "Herbáceo anual",
        "Herbáceo anual",
        "Leñoso permanente",
        "Leñoso permanente",
        "Leñoso permanente"
    ]
})

display(tipo_cultivo)

In [ ]:
matriz_candidatos = (
    eurocrops_candidatos
    .merge(
        informacion_mapa,
        on="cultivo",
        how="left"
    )
    .merge(
        tipo_cultivo,
        on="cultivo",
        how="left"
    )
)

print("Número de cultivos evaluados:", len(matriz_candidatos))

display(matriz_candidatos)

In [ ]:
# Crear indicadores comparables de disponibilidad espacial en EuroCrops
# NO se elimina ni modifica ningún registro original.

matriz_candidatos_eval = matriz_candidatos.copy()

matriz_candidatos_eval["pct_provincias_>=100"] = (
    matriz_candidatos_eval["provincias_>=_100"] / 50 * 100
)

matriz_candidatos_eval["pct_provincias_>=1000"] = (
    matriz_candidatos_eval["provincias_>=_1000"] / 50 * 100
)

matriz_candidatos_eval["pct_provincias_>=5000"] = (
    matriz_candidatos_eval["provincias_>=_5000"] / 50 * 100
)

matriz_candidatos_eval["pct_provincias_>=10000"] = (
    matriz_candidatos_eval["provincias_>=_10000"] / 50 * 100
)

columnas_mostrar = [
    "cultivo",
    "tipo_agronomico",
    "parcelas_total",
    "provincias_con_presencia",
    "provincias_>=_100",
    "pct_provincias_>=100",
    "provincias_>=_1000",
    "pct_provincias_>=1000",
    "provincias_>=_5000",
    "pct_provincias_>=5000",
    "provincias_>=_10000",
    "pct_provincias_>=10000",
    "correspondencia_eurocrops_mapa"
]

display(
    matriz_candidatos_eval[columnas_mostrar]
    .sort_values("parcelas_total", ascending=False)
    .reset_index(drop=True)
)

# Ficheros MAPA- Produccion

In [ ]:
import os
import pandas as pd


# Ruta raíz REAL del proyecto
RUTA_TFM = "/content/TFM_Rendimiento_cultivos"

RUTA_ORIGINALES = os.path.join(RUTA_TFM, "01_Datos_originales")
RUTA_EUROCROPS = os.path.join(RUTA_ORIGINALES, "EuroCrops")
RUTA_MAPA = os.path.join(RUTA_ORIGINALES, "MAPA")

RUTA_PROCESADOS = os.path.join(RUTA_TFM, "02_Datos_procesados")
RUTA_RESULTADOS = os.path.join(RUTA_TFM, "05_Resultados")

print("Existe TFM:", os.path.exists(RUTA_TFM))
print("Existe EuroCrops:", os.path.exists(RUTA_EUROCROPS))
print("Existe MAPA:", os.path.exists(RUTA_MAPA))
print("Existe Procesados:", os.path.exists(RUTA_PROCESADOS))
print("Existe Resultados:", os.path.exists(RUTA_RESULTADOS))

In [ ]:
import os

ruta_mydrive = "/content/TFM_Rendimiento_cultivos"

print("Carpetas de Mi unidad que contienen 'TFM':\n")

for nombre in sorted(os.listdir(ruta_mydrive)):
    if "tfm" in nombre.lower():
        print(repr(nombre))

In [ ]:
archivos_drive_mapa = sorted(os.listdir(RUTA_MAPA))

print("Archivos encontrados en MAPA:")
for archivo in archivos_drive_mapa:
    print(repr(archivo))

In [ ]:
archivos_mapa = {
    ("common_soft_wheat", 2017): os.path.join(RUTA_MAPA, "exec07_1.2.3_trigo_2017.xlsx"),
    ("common_soft_wheat", 2018): os.path.join(RUTA_MAPA, "exec07_1.2.3_trigo_2018.xlsx"),
    ("common_soft_wheat", 2019): os.path.join(RUTA_MAPA, "7.1.2.3_trigo_2019.xlsx"),
    ("common_soft_wheat", 2020): os.path.join(RUTA_MAPA, "7.1.2.3_trigo_2020.xlsx"),
    ("common_soft_wheat", 2021): os.path.join(RUTA_MAPA, "7.1.2.3_trigo_2021.xlsx"),
    ("common_soft_wheat", 2022): os.path.join(RUTA_MAPA, "7.1.2.3_trigo_2022.xlsx"),
    ("common_soft_wheat", 2023): os.path.join(RUTA_MAPA, "7.1.2.1_trigo_2023.xlsx"),

    ("barley", 2017): os.path.join(RUTA_MAPA, "exec07_1.3.3_cebada_2017.xlsx"),
    ("barley", 2018): os.path.join(RUTA_MAPA, "exec07_1.3.3_cebada_2018.xlsx"),
    ("barley", 2019): os.path.join(RUTA_MAPA, "7.1.3.3_cebada_2019.xlsx"),
    ("barley", 2020): os.path.join(RUTA_MAPA, "7.1.3.3_cebada_2020.xlsx"),
    ("barley", 2021): os.path.join(RUTA_MAPA, "7.1.3.3_cebada_2021.xlsx"),
    ("barley", 2022): os.path.join(RUTA_MAPA, "7.1.3.3_cebada_2022.xlsx"),
    ("barley", 2023): os.path.join(RUTA_MAPA, "7.1.8.1_cebada_2023.xlsx"),
}

print("Archivos configurados:", len(archivos_mapa))

#Cebada

In [ ]:
# -----------------------
# Cebada
# -----------------------

ruta_cebada = os.path.join(
    RUTA_MAPA,
    "7.1.8.1_cebada_2023.xlsx"
)

xls_cebada = pd.ExcelFile(ruta_cebada)

print("Hojas del archivo de cebada:")
print(xls_cebada.sheet_names)

In [ ]:
cebada_raw = pd.read_excel(
    ruta_cebada,
    sheet_name=0,
    header=None
)

print("Dimensiones del Excel:")
print(cebada_raw.shape)

display(cebada_raw.iloc[:20, :15])

# Comprobar automáticamente la estructura de los 8 Excel

In [ ]:
sorted(os.listdir(RUTA_MAPA))

In [ ]:
import pandas as pd
import os

archivos_mapa = {
    "trigo_total": os.path.join(
        RUTA_MAPA,
        "7.1.2.1_Trigo.xlsx"
    ),

    "trigo_invierno": os.path.join(
        RUTA_MAPA,
        "7.1.2.6.2.1_Trigo_invierno.xlsx"
    ),

    "trigo_primavera": os.path.join(
        RUTA_MAPA,
        "7.1.2.6.2.2_Trigo_primavera.xlsx"
    ),

    "cebada": os.path.join(
        RUTA_MAPA,
        "7.1.8.1_Cebada.xlsx"
    ),

    "almendro": os.path.join(
        RUTA_MAPA,
        "7.10.6.1.2_Almendro.xlsx"
    ),

    "vina": os.path.join(
        RUTA_MAPA,
        "7.11.1.1_Vina.xlsx"
    ),

    "olivar": os.path.join(
        RUTA_MAPA,
        "7.12.1.1_Olivar.xlsx"
    ),

    "girasol": os.path.join(
        RUTA_MAPA,
        "7.4.4.8.1_Girasol.xlsx"
    )
}

for nombre, ruta in archivos_mapa.items():

    df_temp = pd.read_excel(
        ruta,
        sheet_name=0,
        header=None
    )

    print("=" * 70)
    print(nombre.upper())
    print("Archivo:", os.path.basename(ruta))
    print("Dimensiones:", df_temp.shape)

    # Mostrar las primeras 12 filas y primeras 15 columnas
    display(df_temp.iloc[:12, :15])

# Almendro

In [ ]:
almendro_raw = pd.read_excel(
    os.path.join(
        RUTA_MAPA,
        "7.10.6.1.2_Almendro.xlsx"
    ),
    header=None
)

print("Dimensiones:", almendro_raw.shape)

# Mostrar las filas de cabecera y TODAS las columnas
display(almendro_raw.iloc[:10, :])

# Crear una tabla provincial de rendimientos MAPA

In [ ]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------------
# CONFIGURACIÓN DE LOS ARCHIVOS MAPA
# ---------------------------------------------------------

config_mapa = {
    "barley": {
        "archivo": "/content/TFM_Rendimiento_cultivos/01_Datos_originales/MAPA/7.1.8.1_Cebada.xlsx",
        "fila_inicio": 9,
        "col_provincia": 0,
        "col_rend_secano": 10,
        "col_rend_regadio": 11
    },

    "common_soft_wheat": {
        "archivo": "/content/TFM_Rendimiento_cultivos/01_Datos_originales/MAPA/7.1.2.1_Trigo.xlsx",
        "fila_inicio": 9,
        "col_provincia": 0,
        "col_rend_secano": 10,
        "col_rend_regadio": 11
    },

    "sunflower": {
        "archivo": "/content/TFM_Rendimiento_cultivos/01_Datos_originales/MAPA/7.4.4.8.1_Girasol.xlsx",
        "fila_inicio": 10,
        "col_provincia": 0,
        "col_rend_secano": 10,
        "col_rend_regadio": 11
    },

    "vineyards_wine_vine_rebland_grapes": {
        "archivo": "/content/TFM_Rendimiento_cultivos/01_Datos_originales/MAPA/7.11.1.1_Vina.xlsx",
        "fila_inicio": 8,
        "col_provincia": 0,
        "col_rend_secano": 10,
        "col_rend_regadio": 11
    },

    "olive_plantations": {
        "archivo": "/content/TFM_Rendimiento_cultivos/01_Datos_originales/MAPA/7.12.1.1_Olivar.xlsx",
        "fila_inicio": 11,
        "col_provincia": 0,
        "col_rend_secano": 11,
        "col_rend_regadio": 12
    },

    "almond": {
        "archivo": "/content/TFM_Rendimiento_cultivos/01_Datos_originales/MAPA/7.10.6.1.2_Almendro.xlsx",
        "fila_inicio": 10,
        "col_provincia": 0,
        "col_rend_secano": 33,
        "col_rend_regadio": 34
    }
}

print("Cultivos configurados:", len(config_mapa))

for cultivo, cfg in config_mapa.items():
    print(
        cultivo,
        "→",
        os.path.basename(cfg["archivo"]),
        "| secano:", cfg["col_rend_secano"],
        "| regadío:", cfg["col_rend_regadio"]
    )

# Extraer los datos MAPA sin filtrar todavía

In [ ]:
# ============================================================
# CORREGIR RUTAS DE config_mapa PARA USAR GOOGLE DRIVE
# ============================================================

nombres_archivos_mapa = {
    "barley": "7.1.8.1_Cebada.xlsx",
    "common_soft_wheat": "7.1.2.1_Trigo.xlsx",
    "sunflower": "7.4.4.8.1_Girasol.xlsx",
    "vineyards_wine_vine_rebland_grapes": "7.11.1.1_Vina.xlsx",
    "olive_plantations": "7.12.1.1_Olivar.xlsx",
    "almond": "7.10.6.1.2_Almendro.xlsx"
}

# Reemplazar las rutas antiguas por rutas permanentes de Drive
for cultivo in config_mapa:
    config_mapa[cultivo]["archivo"] = os.path.join(
        RUTA_MAPA,
        nombres_archivos_mapa[cultivo]
    )

# Comprobación
for cultivo, cfg in config_mapa.items():
    print(
        cultivo,
        "->",
        cfg["archivo"],
        "->",
        "OK" if os.path.exists(cfg["archivo"]) else "NO ENCONTRADO"
    )

In [ ]:
# ---------------------------------------------------------
# Extracción inicial de rendimientos MAPA
# SIN filtrar filas todavía
# ---------------------------------------------------------

rendimientos_mapa_raw = {}

for cultivo, cfg in config_mapa.items():

    # Leer Excel completo sin asumir encabezados
    df = pd.read_excel(cfg["archivo"], header=None)

    # Extraer desde la primera fila de datos identificada
    datos = df.iloc[
        cfg["fila_inicio"]:,
        [
            cfg["col_provincia"],
            cfg["col_rend_secano"],
            cfg["col_rend_regadio"]
        ]
    ].copy()

    # Renombrar columnas
    datos.columns = [
        "provincia_mapa",
        "rendimiento_secano_kg_ha",
        "rendimiento_regadio_kg_ha"
    ]

    # Conservar índice original del Excel
    datos["fila_excel_original"] = datos.index

    # Añadir cultivo
    datos["cultivo"] = cultivo

    # Reordenar
    datos = datos[
        [
            "cultivo",
            "fila_excel_original",
            "provincia_mapa",
            "rendimiento_secano_kg_ha",
            "rendimiento_regadio_kg_ha"
        ]
    ]

    # Guardar SIN FILTRAR
    rendimientos_mapa_raw[cultivo] = datos


print("Extracción terminada.")
print("Cultivos extraídos:", len(rendimientos_mapa_raw))

In [ ]:
# Inspeccionar qué filas he extraído realmente antes de filtrar

for cultivo, datos in rendimientos_mapa_raw.items():

    print("\n" + "=" * 80)
    print(cultivo.upper())
    print("Número de filas extraídas:", len(datos))
    print("=" * 80)

    print("\nPRIMERAS 12 FILAS:")
    display(datos.head(12))

    print("\nÚLTIMAS 12 FILAS:")
    display(datos.tail(12))

In [ ]:
# ============================================================
# Identificar exclusivamente las 50 provincias españolas
# SIN modificar los datos originales
# ============================================================

provincias_50 = [
    "A Coruña", "Álava", "Albacete", "Alicante", "Almería",
    "Asturias", "Ávila", "Badajoz", "Barcelona", "Bizkaia",
    "Burgos", "Cáceres", "Cádiz", "Cantabria", "Castellón",
    "Ciudad Real", "Córdoba", "Cuenca", "Gipuzkoa", "Girona",
    "Granada", "Guadalajara", "Huelva", "Huesca", "Illes Balears",
    "Jaén", "La Rioja", "Las Palmas", "León", "Lleida",
    "Lugo", "Madrid", "Málaga", "Murcia", "Navarra",
    "Ourense", "Palencia", "Pontevedra", "Salamanca",
    "Santa Cruz de Tenerife", "Segovia", "Sevilla", "Soria",
    "Tarragona", "Teruel", "Toledo", "Valencia", "Valladolid",
    "Zamora", "Zaragoza"
]

print("Número de provincias de referencia:", len(provincias_50))

In [ ]:
# ============================================================
# Auditoría de nombres geográficos presentes en MAPA
# NO modifica ni filtra los datos originales
# ============================================================

import pandas as pd

# Reunimos los datos extraídos de los 6 cultivos
mapa_todos_raw = pd.concat(
    rendimientos_mapa_raw.values(),
    ignore_index=True
)

# Nombres únicos encontrados en la columna provincia_mapa
nombres_mapa = (
    mapa_todos_raw["provincia_mapa"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Número de nombres geográficos distintos encontrados en MAPA:",
      len(nombres_mapa))

print("\nNOMBRES PRESENTES EN MAPA QUE NO COINCIDEN EXACTAMENTE")
print("CON NUESTRA LISTA DE 50 PROVINCIAS:\n")

no_coinciden = [
    nombre
    for nombre in nombres_mapa
    if nombre not in provincias_50
]

for nombre in no_coinciden:
    print(repr(nombre))

print("\nTotal de nombres sin coincidencia exacta:", len(no_coinciden))

In [ ]:
# ============================================================
# Crear equivalencias entre nombres MAPA y provincias estándar
# NO elimina filas
# NO modifica provincia_mapa
# ============================================================

# Primero comprobamos cómo están escritos esos nombres
# en nuestra lista oficial provincias_50

for p in provincias_50:
    if any(x in p.lower() for x in [
        "astur", "murcia", "tenerife",
        "vizc", "bizk", "guip", "gipuz"
    ]):
        print(repr(p))

In [ ]:
# ============================================================
# Crear provincia_estandar
# Conservamos provincia_mapa intacta
# ============================================================

equivalencias_provincias = {
    "P. DE ASTURIAS": "Asturias",
    "R. DE MURCIA": "Murcia",
    "S.C. de Tenerife": "Santa Cruz de Tenerife",
    "Vizcaya": "Bizkaia",
    "Guipúzcoa": "Gipuzkoa"
}

# Trabajamos sobre una COPIA
mapa_todos_audit = mapa_todos_raw.copy()

# Conservamos el nombre original y creamos una nueva columna
mapa_todos_audit["provincia_estandar"] = (
    mapa_todos_audit["provincia_mapa"]
    .replace(equivalencias_provincias)
)

print("Equivalencias aplicadas:\n")

for original, estandar in equivalencias_provincias.items():
    n = (
        mapa_todos_audit["provincia_mapa"]
        .eq(original)
        .sum()
    )

    print(f"{original} -> {estandar} | filas encontradas: {n}")

In [ ]:
# ============================================================
# Comprobar cobertura de las 50 provincias
# ============================================================

provincias_en_mapa = set(
    mapa_todos_audit.loc[
        mapa_todos_audit["provincia_estandar"].isin(provincias_50),
        "provincia_estandar"
    ]
)

faltan = sorted(set(provincias_50) - provincias_en_mapa)
extras = sorted(provincias_en_mapa - set(provincias_50))

print("Provincias de referencia:", len(provincias_50))
print("Provincias identificadas en MAPA:", len(provincias_en_mapa))

print("\nProvincias que faltan:")
print(faltan)

print("\nProvincias extra:")
print(extras)

In [ ]:
# ============================================================
# Diagnóstico exacto de provincias uniprovinciales y nombres
# problemáticos
# NO modifica ningún dato
# ============================================================

terminos_busqueda = [
    "ASTURIAS",
    "MURCIA",
    "CANTABRIA",
    "BALEARES",
    "RIOJA",
    "MADRID",
    "NAVARRA"
]

print("NOMBRES EXACTOS ENCONTRADOS EN provincia_mapa:\n")

valores_unicos = (
    mapa_todos_raw["provincia_mapa"]
    .dropna()
    .astype(str)
    .unique()
)

for termino in terminos_busqueda:

    encontrados = [
        x for x in valores_unicos
        if termino in x.upper()
    ]

    print(f"\n--- {termino} ---")

    for x in encontrados:
        print(
            repr(x),
            "| longitud:", len(x),
            "| filas:", (mapa_todos_raw["provincia_mapa"] == x).sum()
        )

In [ ]:
# ============================================================
# Normalización controlada de nombres provinciales MAPA
# provincia_mapa permanece intacta
# ============================================================

mapa_todos_audit = mapa_todos_raw.copy()

# 1. Crear nueva columna eliminando espacios iniciales/finales
mapa_todos_audit["provincia_estandar"] = (
    mapa_todos_audit["provincia_mapa"]
    .astype("string")
    .str.strip()
)

# 2. Equivalencias reales MAPA -> nomenclatura provincias_50
equivalencias_provincias = {
    "P. DE ASTURIAS": "Asturias",
    "R. DE MURCIA": "Murcia",
    "S.C. de Tenerife": "Santa Cruz de Tenerife",
    "Vizcaya": "Bizkaia",
    "Guipúzcoa": "Gipuzkoa",

    # Provincias uniprovinciales / diferencias de nomenclatura
    "CANTABRIA": "Cantabria",
    "BALEARES": "Illes Balears",
    "LA RIOJA": "La Rioja",
    "MADRID": "Madrid",
    "NAVARRA": "Navarra"
}

mapa_todos_audit["provincia_estandar"] = (
    mapa_todos_audit["provincia_estandar"]
    .replace(equivalencias_provincias)
)

print("Normalización realizada.")

In [ ]:
# ============================================================
# Verificación final de cobertura provincial
# ============================================================

provincias_en_mapa = set(
    mapa_todos_audit.loc[
        mapa_todos_audit["provincia_estandar"].isin(provincias_50),
        "provincia_estandar"
    ]
)

faltan = sorted(set(provincias_50) - provincias_en_mapa)

print("Provincias de referencia:", len(provincias_50))
print("Provincias identificadas en MAPA:", len(provincias_en_mapa))

print("\nProvincias que faltan:")
print(faltan)

In [ ]:
# ============================================================
# Auditoría de cobertura provincial del rendimiento MAPA
# NO selecciona ni elimina cultivos
# ============================================================

# Nos quedamos EN UNA COPIA con las filas que corresponden
# a las 50 provincias ya validadas
mapa_provincias_audit = mapa_todos_audit[
    mapa_todos_audit["provincia_estandar"].isin(provincias_50)
].copy()

resumen_rendimiento = []

for cultivo, datos in mapa_provincias_audit.groupby("cultivo"):

    secano = pd.to_numeric(
        datos["rendimiento_secano_kg_ha"],
        errors="coerce"
    )

    regadio = pd.to_numeric(
        datos["rendimiento_regadio_kg_ha"],
        errors="coerce"
    )

    fila = {
        "cultivo": cultivo,

        # Comprobación territorial
        "provincias": datos["provincia_estandar"].nunique(),

        # SECANO
        "secano_con_dato": secano.notna().sum(),
        "secano_positivo": (secano > 0).sum(),
        "secano_cero": (secano == 0).sum(),

        # REGADÍO
        "regadio_con_dato": regadio.notna().sum(),
        "regadio_positivo": (regadio > 0).sum(),
        "regadio_cero": (regadio == 0).sum(),
    }

    resumen_rendimiento.append(fila)

resumen_rendimiento_df = pd.DataFrame(resumen_rendimiento)

display(
    resumen_rendimiento_df.sort_values(
        "secano_positivo",
        ascending=False
    ).reset_index(drop=True)
)

In [ ]:
print(cultivos_por_provincia_df.columns.tolist())

In [ ]:
display(
    cultivos_por_provincia_df[
        [
            "codigo_provincia",
            "cultivo_resumen",
            "numero_registros"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# Diagnóstico de codigo_provincia
# NO modifica los datos
# ============================================================

codigos_raw = cultivos_por_provincia_df["codigo_provincia"]

print("Tipo de datos de la columna:")
print(codigos_raw.dtype)

print("\nNúmero de valores distintos:")
print(codigos_raw.nunique(dropna=False))

print("\nValores distintos encontrados:")
print(codigos_raw.drop_duplicates().tolist())

print("\nTipos Python encontrados:")
print(
    codigos_raw
    .dropna()
    .map(type)
    .value_counts()
)

In [ ]:
# ============================================================
# Normalizar codigo_provincia
# Conservamos codigo_provincia original
# ============================================================

cultivos_por_provincia_audit = cultivos_por_provincia_df.copy()

cultivos_por_provincia_audit["codigo_provincia_std"] = pd.to_numeric(
    cultivos_por_provincia_audit["codigo_provincia"],
    errors="coerce"
).astype("Int64")

print(
    "Códigos originales distintos:",
    cultivos_por_provincia_audit["codigo_provincia"].nunique()
)

print(
    "Códigos normalizados distintos:",
    cultivos_por_provincia_audit["codigo_provincia_std"].nunique()
)

print(
    "Valores que no pudieron convertirse:",
    cultivos_por_provincia_audit["codigo_provincia_std"].isna().sum()
)

print("\nCódigos normalizados:")
print(
    sorted(
        cultivos_por_provincia_audit[
            "codigo_provincia_std"
        ].dropna().unique().tolist()
    )
)

In [ ]:
# ============================================================
# Relación código provincial normalizado <-> archivo
# ============================================================

mapa_codigos_archivos = (
    cultivos_por_provincia_audit[
        ["codigo_provincia_std", "archivo"]
    ]
    .drop_duplicates()
    .sort_values("codigo_provincia_std")
    .reset_index(drop=True)
)

print(
    "Número de códigos provinciales:",
    mapa_codigos_archivos["codigo_provincia_std"].nunique()
)

print(
    "Número de combinaciones código-archivo:",
    len(mapa_codigos_archivos)
)

display(mapa_codigos_archivos)

In [ ]:
# ============================================================
# Crear equivalencia codigo_provincia_std -> provincia_estandar
# ============================================================

codigo_a_provincia = {
    1: "Álava",
    2: "Albacete",
    3: "Alicante",
    4: "Almería",
    5: "Ávila",
    6: "Badajoz",
    7: "Illes Balears",
    8: "Barcelona",
    9: "Burgos",
    10: "Cáceres",
    11: "Cádiz",
    12: "Castellón",
    13: "Ciudad Real",
    14: "Córdoba",
    15: "A Coruña",
    16: "Cuenca",
    17: "Girona",
    18: "Granada",
    19: "Guadalajara",
    20: "Gipuzkoa",
    21: "Huelva",
    22: "Huesca",
    23: "Jaén",
    24: "León",
    25: "Lleida",
    26: "La Rioja",
    27: "Lugo",
    28: "Madrid",
    29: "Málaga",
    30: "Murcia",
    31: "Navarra",
    32: "Ourense",
    33: "Asturias",
    34: "Palencia",
    35: "Las Palmas",
    36: "Pontevedra",
    37: "Salamanca",
    38: "Santa Cruz de Tenerife",
    39: "Cantabria",
    40: "Segovia",
    41: "Sevilla",
    42: "Soria",
    43: "Tarragona",
    44: "Teruel",
    45: "Toledo",
    46: "Valencia",
    47: "Valladolid",
    48: "Bizkaia",
    49: "Zamora",
    50: "Zaragoza"
}

print("Número de equivalencias:", len(codigo_a_provincia))

In [ ]:
#----------------------
# Añadir provincia a tabla crops
#----------------------

cultivos_por_provincia_audit["provincia_estandar"] = (
    cultivos_por_provincia_audit["codigo_provincia_std"]
    .map(codigo_a_provincia)
)

print(
    "Provincias sin nombre asignado:",
    cultivos_por_provincia_audit["provincia_estandar"].isna().sum()
)

print(
    "Número de provincias distintas:",
    cultivos_por_provincia_audit["provincia_estandar"].nunique()
)

In [ ]:
display(
    cultivos_por_provincia_audit[
        [
            "codigo_provincia_std",
            "provincia_estandar",
            "archivo"
        ]
    ]
    .drop_duplicates()
    .sort_values("codigo_provincia_std")
    .reset_index(drop=True)
)

In [ ]:
print("COLUMNAS EUROCROPS:")
print(cultivos_por_provincia_audit.columns.tolist())

print("\nCOLUMNAS MAPA:")
print(mapa_provincias_audit.columns.tolist())

In [ ]:
# Mostrar variables tipo DataFrame disponibles en memoria

for nombre, variable in list(globals().items()):
    if hasattr(variable, "columns"):
        print(nombre, "->", variable.shape)

#cruce EuroCrops x MAPA


In [ ]:
# ============================================================
# Cruce EuroCrops + MAPA por cultivo y provincia
# ============================================================

euro = cultivos_por_provincia_audit.copy()

# Crear columna de cultivo con el mismo nombre que en MAPA
euro["cultivo"] = euro["cultivo_resumen"]

mapa = mapa_provincias_audit.copy()

# Asegurar que los rendimientos son numéricos
mapa["rendimiento_secano_kg_ha"] = pd.to_numeric(
    mapa["rendimiento_secano_kg_ha"],
    errors="coerce"
)

mapa["rendimiento_regadio_kg_ha"] = pd.to_numeric(
    mapa["rendimiento_regadio_kg_ha"],
    errors="coerce"
)

# Nos quedamos solo con las columnas necesarias de MAPA
mapa_merge = mapa[
    [
        "cultivo",
        "provincia_estandar",
        "rendimiento_secano_kg_ha",
        "rendimiento_regadio_kg_ha"
    ]
].copy()

# Cruce por cultivo + provincia
cruce_ec_mapa = euro.merge(
    mapa_merge,
    on=["cultivo", "provincia_estandar"],
    how="left",
    validate="many_to_one"
)

print("Filas EuroCrops antes del cruce:", len(euro))
print("Filas después del cruce:", len(cruce_ec_mapa))

print(
    "Filas con algún rendimiento MAPA:",
    (
        cruce_ec_mapa["rendimiento_secano_kg_ha"].notna()
        |
        cruce_ec_mapa["rendimiento_regadio_kg_ha"].notna()
    ).sum()
)

print(
    "Cultivos presentes:",
    cruce_ec_mapa["cultivo"].nunique()
)

print(
    "Provincias presentes:",
    cruce_ec_mapa["provincia_estandar"].nunique()
)

In [ ]:
display(
    cruce_ec_mapa[
        [
            "cultivo",
            "provincia_estandar",
            "numero_registros",
            "superficie_total_par_supcul",
            "rendimiento_secano_kg_ha",
            "rendimiento_regadio_kg_ha"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# Cobertura conjunta EuroCrops + MAPA para los 6 candidatos
# NO modifica ni filtra los DataFrames originales
# ============================================================

candidatos = [
    "barley",
    "common_soft_wheat",
    "sunflower",
    "olive_plantations",
    "vineyards_wine_vine_rebland_grapes",
    "almond"
]

evaluacion_conjunta = cruce_ec_mapa[
    cruce_ec_mapa["cultivo"].isin(candidatos)
].copy()

# Indicadores
evaluacion_conjunta["mapa_secano_positivo"] = (
    evaluacion_conjunta["rendimiento_secano_kg_ha"] > 0
)

evaluacion_conjunta["mapa_regadio_positivo"] = (
    evaluacion_conjunta["rendimiento_regadio_kg_ha"] > 0
)

evaluacion_conjunta["mapa_algun_rendimiento_positivo"] = (
    evaluacion_conjunta["mapa_secano_positivo"]
    | evaluacion_conjunta["mapa_regadio_positivo"]
)

# Resumen por cultivo
resumen_cruce = (
    evaluacion_conjunta
    .groupby("cultivo")
    .agg(
        provincias_eurocrops=("provincia_estandar", "nunique"),
        parcelas_total=("numero_registros", "sum"),
        superficie_total=("superficie_total_par_supcul", "sum"),

        provincias_mapa_secano_positivo=(
            "mapa_secano_positivo", "sum"
        ),

        provincias_mapa_regadio_positivo=(
            "mapa_regadio_positivo", "sum"
        ),

        provincias_mapa_algun_rendimiento_positivo=(
            "mapa_algun_rendimiento_positivo", "sum"
        )
    )
    .reset_index()
)

display(resumen_cruce)

In [ ]:
# ============================================================
# Cobertura útil conjunta EuroCrops + MAPA
# ============================================================

umbrales = [100, 500, 1000, 5000, 10000]

resultados = []

for cultivo in candidatos:

    d = evaluacion_conjunta[
        evaluacion_conjunta["cultivo"] == cultivo
    ].copy()

    fila = {
        "cultivo": cultivo,
        "parcelas_total": d["numero_registros"].sum()
    }

    for u in umbrales:

        # Provincia suficientemente representada en EuroCrops
        euro_ok = d["numero_registros"] >= u

        # MAPA tiene al menos un rendimiento > 0
        mapa_ok = (
            (d["rendimiento_secano_kg_ha"] > 0)
            |
            (d["rendimiento_regadio_kg_ha"] > 0)
        )

        # Ambas condiciones simultáneamente
        fila[f"provincias_utiles_>={u}"] = (
            euro_ok & mapa_ok
        ).sum()

    resultados.append(fila)

cobertura_util_df = pd.DataFrame(resultados)

cobertura_util_df = cobertura_util_df.sort_values(
    "provincias_utiles_>=1000",
    ascending=False
).reset_index(drop=True)

display(cobertura_util_df)

In [ ]:
# ============================================================
# Concentración territorial de EuroCrops por cultivo
# NO modifica los datos originales
# ============================================================

resultados_concentracion = []

for cultivo in candidatos:

    d = evaluacion_conjunta[
        evaluacion_conjunta["cultivo"] == cultivo
    ].copy()

    # Solo provincias con presencia real del cultivo
    d = d[d["numero_registros"] > 0].copy()

    total_parcelas = d["numero_registros"].sum()
    total_superficie = d["superficie_total_par_supcul"].sum()

    # Ordenar de mayor a menor representación
    d = d.sort_values("numero_registros", ascending=False)

    top1 = d["numero_registros"].head(1).sum()
    top3 = d["numero_registros"].head(3).sum()
    top5 = d["numero_registros"].head(5).sum()
    top10 = d["numero_registros"].head(10).sum()

    resultados_concentracion.append({
        "cultivo": cultivo,
        "provincias_con_presencia": len(d),
        "parcelas_total": total_parcelas,
        "superficie_total": total_superficie,
        "pct_parcelas_top1": round(100 * top1 / total_parcelas, 2),
        "pct_parcelas_top3": round(100 * top3 / total_parcelas, 2),
        "pct_parcelas_top5": round(100 * top5 / total_parcelas, 2),
        "pct_parcelas_top10": round(100 * top10 / total_parcelas, 2)
    })

concentracion_df = pd.DataFrame(resultados_concentracion)

display(
    concentracion_df.sort_values(
        "pct_parcelas_top5",
        ascending=True
    ).reset_index(drop=True)
)

In [ ]:
# ============================================================
# Ranking provincial EuroCrops + rendimiento MAPA
# NO modifica los datos originales
# ============================================================

for cultivo in candidatos:

    d = evaluacion_conjunta[
        evaluacion_conjunta["cultivo"] == cultivo
    ].copy()

    # Solo provincias donde EuroCrops tiene presencia
    d = d[d["numero_registros"] > 0]

    # Ordenar por número de parcelas
    d = d.sort_values(
        "numero_registros",
        ascending=False
    )

    print("\n" + "=" * 80)
    print(cultivo.upper())
    print("=" * 80)

    display(
        d[
            [
                "provincia_estandar",
                "numero_registros",
                "superficie_total_par_supcul",
                "rendimiento_secano_kg_ha",
                "rendimiento_regadio_kg_ha"
            ]
        ].head(15)
    )

In [ ]:
# ============================================================
# Variabilidad provincial del rendimiento MAPA
# ============================================================

import pandas as pd
import numpy as np

resultados_variabilidad = []

for cultivo in candidatos:

    d = evaluacion_conjunta[
        evaluacion_conjunta["cultivo"] == cultivo
    ].copy()

    # Solo provincias donde EuroCrops tiene presencia
    d = d[d["numero_registros"] > 0]

    for sistema, columna in [
        ("secano", "rendimiento_secano_kg_ha"),
        ("regadio", "rendimiento_regadio_kg_ha")
    ]:

        # Para este análisis consideramos rendimiento positivo
        valores = pd.to_numeric(d[columna], errors="coerce")
        valores = valores[valores > 0]

        if len(valores) > 0:
            resultados_variabilidad.append({
                "cultivo": cultivo,
                "sistema": sistema,
                "n_provincias": len(valores),
                "media": valores.mean(),
                "mediana": valores.median(),
                "std": valores.std(),
                "min": valores.min(),
                "p25": valores.quantile(0.25),
                "p75": valores.quantile(0.75),
                "max": valores.max(),
                "cv_pct": (valores.std() / valores.mean()) * 100
            })

variabilidad_rendimiento = pd.DataFrame(resultados_variabilidad)

# Redondear únicamente para visualizar
columnas_numericas = [
    "media", "mediana", "std", "min",
    "p25", "p75", "max", "cv_pct"
]

variabilidad_rendimiento[columnas_numericas] = (
    variabilidad_rendimiento[columnas_numericas].round(1)
)

display(variabilidad_rendimiento)

In [ ]:
# ============================================================
# Identificación de rendimientos provinciales potencialmente atípicos
# Método IQR
# IMPORTANTE: aquí NO eliminamos ningún dato
# ============================================================

import pandas as pd
import numpy as np

outliers_resultado = []

for cultivo in candidatos:

    d = evaluacion_conjunta[
        evaluacion_conjunta["cultivo"] == cultivo
    ].copy()

    # Solo provincias donde EuroCrops tiene presencia
    d = d[d["numero_registros"] > 0]

    for sistema, columna in [
        ("secano", "rendimiento_secano_kg_ha"),
        ("regadio", "rendimiento_regadio_kg_ha")
    ]:

        d_temp = d[
            ["provincia_estandar", "numero_registros",
             "superficie_total_par_supcul", columna]
        ].copy()

        d_temp[columna] = pd.to_numeric(
            d_temp[columna], errors="coerce"
        )

        # Solo rendimientos positivos
        d_temp = d_temp[d_temp[columna] > 0].copy()

        if len(d_temp) < 4:
            continue

        q1 = d_temp[columna].quantile(0.25)
        q3 = d_temp[columna].quantile(0.75)
        iqr = q3 - q1

        limite_inferior = q1 - 1.5 * iqr
        limite_superior = q3 + 1.5 * iqr

        d_out = d_temp[
            (d_temp[columna] < limite_inferior) |
            (d_temp[columna] > limite_superior)
        ].copy()

        for _, fila in d_out.iterrows():

            outliers_resultado.append({
                "cultivo": cultivo,
                "sistema": sistema,
                "provincia": fila["provincia_estandar"],
                "rendimiento_kg_ha": fila[columna],
                "numero_registros_eurocrops": fila["numero_registros"],
                "superficie_eurocrops": fila["superficie_total_par_supcul"],
                "limite_inferior_iqr": limite_inferior,
                "limite_superior_iqr": limite_superior
            })


outliers_rendimiento = pd.DataFrame(outliers_resultado)

if len(outliers_rendimiento) == 0:
    print("No se han detectado valores potencialmente atípicos mediante IQR.")
else:
    outliers_rendimiento = (
        outliers_rendimiento
        .sort_values(
            ["cultivo", "sistema", "rendimiento_kg_ha"],
            ascending=[True, True, False]
        )
        .reset_index(drop=True)
    )

    display(outliers_rendimiento.round(1))

    print("\nNúmero de posibles outliers por cultivo y sistema:")

    display(
        outliers_rendimiento
        .groupby(["cultivo", "sistema"])
        .size()
        .reset_index(name="n_posibles_outliers")
    )

In [ ]:
# ============================================================
# Tabla integrada para evaluación de cultivos candidatos
# ============================================================

import pandas as pd
import numpy as np

resultados = []

for cultivo in candidatos:

    d = evaluacion_conjunta[
        evaluacion_conjunta["cultivo"] == cultivo
    ].copy()

    # Provincias donde el cultivo existe en EuroCrops
    d = d[d["numero_registros"] > 0].copy()

    n_provincias = d["provincia_estandar"].nunique()
    parcelas = d["numero_registros"].sum()
    superficie = d["superficie_total_par_supcul"].sum()

    secano = pd.to_numeric(
        d["rendimiento_secano_kg_ha"], errors="coerce"
    )

    regadio = pd.to_numeric(
        d["rendimiento_regadio_kg_ha"], errors="coerce"
    )

    n_secano = (secano > 0).sum()
    n_regadio = (regadio > 0).sum()
    n_algun_rend = ((secano > 0) | (regadio > 0)).sum()

    # Provincias con tamaño EuroCrops relevante
    n_100 = (d["numero_registros"] >= 100).sum()
    n_500 = (d["numero_registros"] >= 500).sum()
    n_1000 = (d["numero_registros"] >= 1000).sum()
    n_5000 = (d["numero_registros"] >= 5000).sum()
    n_10000 = (d["numero_registros"] >= 10000).sum()

    # Concentración territorial
    registros_ordenados = (
        d["numero_registros"]
        .sort_values(ascending=False)
        .reset_index(drop=True)
    )

    pct_top1 = (
        registros_ordenados.iloc[:1].sum() / parcelas * 100
        if parcelas > 0 else np.nan
    )

    pct_top5 = (
        registros_ordenados.iloc[:5].sum() / parcelas * 100
        if parcelas > 0 else np.nan
    )

    pct_top10 = (
        registros_ordenados.iloc[:10].sum() / parcelas * 100
        if parcelas > 0 else np.nan
    )

    # Variabilidad del rendimiento
    sec_pos = secano[secano > 0]
    reg_pos = regadio[regadio > 0]

    cv_secano = (
        sec_pos.std() / sec_pos.mean() * 100
        if len(sec_pos) > 1 else np.nan
    )

    cv_regadio = (
        reg_pos.std() / reg_pos.mean() * 100
        if len(reg_pos) > 1 else np.nan
    )

    resultados.append({
        "cultivo": cultivo,
        "provincias_eurocrops": n_provincias,
        "parcelas_total": parcelas,
        "superficie_total": superficie,

        "prov_rend_secano": n_secano,
        "prov_rend_regadio": n_regadio,
        "prov_algun_rend": n_algun_rend,

        "prov_>=1000_parcelas": n_1000,
        "prov_>=5000_parcelas": n_5000,
        "prov_>=10000_parcelas": n_10000,

        "pct_top1": pct_top1,
        "pct_top5": pct_top5,
        "pct_top10": pct_top10,

        "cv_secano_pct": cv_secano,
        "cv_regadio_pct": cv_regadio
    })


comparativa_final_candidatos = pd.DataFrame(resultados)

# Redondeo solo para visualización
cols_round = [
    "pct_top1",
    "pct_top5",
    "pct_top10",
    "cv_secano_pct",
    "cv_regadio_pct"
]

comparativa_final_candidatos[cols_round] = (
    comparativa_final_candidatos[cols_round].round(1)
)

display(
    comparativa_final_candidatos
    .sort_values(
        ["prov_>=10000_parcelas", "prov_algun_rend", "parcelas_total"],
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# COMPROBACIÓN SEMÁNTICA DEL ARCHIVO MAPA DE TRIGO
# ============================================================

import pandas as pd
import os

archivo_trigo = os.path.join(
    RUTA_MAPA,
    "7.1.2.1_Trigo.xlsx"
)

trigo_raw = pd.read_excel(
    archivo_trigo,
    header=None
)

print("Dimensiones:", trigo_raw.shape)

# Primeras filas para identificar exactamente
# qué categoría de trigo contiene el fichero
display(trigo_raw.head(15))

# DECISIÓN - haré modelos para Trigo y Cebada

# ULTIMA COMPROBACIÖN

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# DATASET BASE PARA LOS DOS CULTIVOS SELECCIONADOS
# EuroCrops + MAPA
# ============================================================

cultivos_finales = [
    "common_soft_wheat",
    "barley"
]

# Usamos el DataFrame YA cruzado EuroCrops + MAPA
base_modelo = cruce_ec_mapa[
    cruce_ec_mapa["cultivo"].isin(cultivos_finales)
].copy()

# Variables que nos interesan
columnas = [
    "cultivo",
    "provincia_estandar",
    "numero_registros",
    "superficie_total_par_supcul",
    "rendimiento_secano_kg_ha",
    "rendimiento_regadio_kg_ha"
]

base_modelo = base_modelo[columnas].copy()

# ------------------------------------------------------------
# 1. Comprobar duplicados cultivo × provincia
# ------------------------------------------------------------

duplicados = base_modelo.duplicated(
    subset=["cultivo", "provincia_estandar"]
).sum()

print("Número de filas:", len(base_modelo))
print("Duplicados cultivo × provincia:", duplicados)

# ------------------------------------------------------------
# 2. Provincias por cultivo
# ------------------------------------------------------------

print("\nNúmero de provincias por cultivo:")

display(
    base_modelo
    .groupby("cultivo")["provincia_estandar"]
    .nunique()
)

# ------------------------------------------------------------
# 3. Disponibilidad del rendimiento
# ------------------------------------------------------------

resumen = (
    base_modelo
    .groupby("cultivo")
    .agg(
        provincias=(
            "provincia_estandar",
            "nunique"
        ),

        rendimiento_secano_positivo=(
            "rendimiento_secano_kg_ha",
            lambda x: (x > 0).sum()
        ),

        rendimiento_regadio_positivo=(
            "rendimiento_regadio_kg_ha",
            lambda x: (x > 0).sum()
        ),

        parcelas_total=(
            "numero_registros",
            "sum"
        ),

        superficie_total=(
            "superficie_total_par_supcul",
            "sum"
        )
    )
)

print("\nResumen:")
display(resumen)

# ------------------------------------------------------------
# 4. Visualizar primeras filas
# ------------------------------------------------------------

print("\nPrimeras filas:")

display(
    base_modelo
    .sort_values(
        ["cultivo", "provincia_estandar"]
    )
    .head(20)
)

# ADJUNTO datos temporales del 2017 hasta el 2023 para los cultivos de trigo y cebada


In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# ARCHIVOS MAPA: TRIGO Y CEBADA 2017-2023
# RUTAS PERMANENTES EN GOOGLE DRIVE
# ============================================================

archivos_mapa = {
    ("common_soft_wheat", 2017): os.path.join(
        RUTA_MAPA, "exec07_1.2.3_trigo_2017.xlsx"
    ),
    ("common_soft_wheat", 2018): os.path.join(
        RUTA_MAPA, "exec07_1.2.3_trigo_2018.xlsx"
    ),
    ("common_soft_wheat", 2019): os.path.join(
        RUTA_MAPA, "7.1.2.3_trigo_2019.xlsx"
    ),
    ("common_soft_wheat", 2020): os.path.join(
        RUTA_MAPA, "7.1.2.3_trigo_2020.xlsx"
    ),
    ("common_soft_wheat", 2021): os.path.join(
        RUTA_MAPA, "7.1.2.3_trigo_2021.xlsx"
    ),
    ("common_soft_wheat", 2022): os.path.join(
        RUTA_MAPA, "7.1.2.3_trigo_2022.xlsx"
    ),
    ("common_soft_wheat", 2023): os.path.join(
        RUTA_MAPA, "7.1.2.1_trigo_2023.xlsx"
    ),

    ("barley", 2017): os.path.join(
        RUTA_MAPA, "exec07_1.3.3_cebada_2017.xlsx"
    ),
    ("barley", 2018): os.path.join(
        RUTA_MAPA, "exec07_1.3.3_cebada_2018.xlsx"
    ),
    ("barley", 2019): os.path.join(
        RUTA_MAPA, "7.1.3.3_cebada_2019.xlsx"
    ),
    ("barley", 2020): os.path.join(
        RUTA_MAPA, "7.1.3.3_cebada_2020.xlsx"
    ),
    ("barley", 2021): os.path.join(
        RUTA_MAPA, "7.1.3.3_cebada_2021.xlsx"
    ),
    ("barley", 2022): os.path.join(
        RUTA_MAPA, "7.1.3.3_cebada_2022.xlsx"
    ),
    ("barley", 2023): os.path.join(
        RUTA_MAPA, "7.1.8.1_cebada_2023.xlsx"
    )
}

print("Número de archivos configurados:", len(archivos_mapa))

# Comprobar que existen
faltantes = []

for (cultivo, ano), ruta in archivos_mapa.items():
    estado = "OK" if os.path.exists(ruta) else "NO ENCONTRADO"

    print(
        f"{ano} | {cultivo:18s} | "
        f"{os.path.basename(ruta)} | {estado}"
    )

    if not os.path.exists(ruta):
        faltantes.append(ruta)

print("\nArchivos no encontrados:", len(faltantes))

In [ ]:
# ============================================================
# Detectar estructura de los 14 Excel MAPA
# ============================================================

import pandas as pd

estructura_mapa = []

for (cultivo, ano), ruta in archivos_mapa.items():

    df_tmp = pd.read_excel(ruta, header=None)

    # Buscar texto relacionado con provincia y rendimiento
    coincidencias_provincia = []
    coincidencias_rendimiento = []

    for fila in range(df_tmp.shape[0]):
        for col in range(df_tmp.shape[1]):

            valor = df_tmp.iat[fila, col]

            if pd.isna(valor):
                continue

            texto = str(valor).strip().lower()

            if "provincias" in texto:
                coincidencias_provincia.append((fila, col, str(valor)))

            if "rendimiento" in texto:
                coincidencias_rendimiento.append((fila, col, str(valor)))

    estructura_mapa.append({
        "cultivo": cultivo,
        "ano": ano,
        "filas": df_tmp.shape[0],
        "columnas": df_tmp.shape[1],
        "provincia_encontrada": coincidencias_provincia[:3],
        "rendimiento_encontrado": coincidencias_rendimiento[:5]
    })

estructura_mapa_df = pd.DataFrame(estructura_mapa)

display(estructura_mapa_df)

In [ ]:
# ============================================================
# VER ESTRUCTURA REAL 2017-2022
# Solo necesitamos inspeccionar un archivo representativo
# ============================================================

archivo_ejemplo = archivos_mapa[("common_soft_wheat", 2022)]

tmp = pd.read_excel(archivo_ejemplo, header=None)

# Mostrar las primeras 12 filas completas
display(tmp.iloc[:12, :])

In [ ]:
# ============================================================
# CONSTRUIR BASE DE RENDIMIENTOS MAPA 2017-2023
# Trigo + cebada
# ============================================================

import pandas as pd
import numpy as np

resultados = []

for (cultivo, ano), ruta in archivos_mapa.items():

    raw = pd.read_excel(ruta, header=None)

    # Estructura MAPA 2017-2022
    if ano <= 2022:
        datos = raw.iloc[7:, [0, 4, 5]].copy()
        datos.columns = [
            "provincia_mapa",
            "rendimiento_secano_kg_ha",
            "rendimiento_regadio_kg_ha"
        ]

    # Estructura MAPA 2023
    else:
        datos = raw.iloc[9:, [0, 10, 11]].copy()
        datos.columns = [
            "provincia_mapa",
            "rendimiento_secano_kg_ha",
            "rendimiento_regadio_kg_ha"
        ]

    # Añadir cultivo y año
    datos["cultivo"] = cultivo
    datos["ano"] = ano

    # Convertir rendimientos a numérico.
    # Símbolos como "-" pasarán a NaN.
    datos["rendimiento_secano_kg_ha"] = pd.to_numeric(
        datos["rendimiento_secano_kg_ha"],
        errors="coerce"
    )

    datos["rendimiento_regadio_kg_ha"] = pd.to_numeric(
        datos["rendimiento_regadio_kg_ha"],
        errors="coerce"
    )

    resultados.append(datos)

# Unir los 14 Excel
rendimientos_mapa_2017_2023 = pd.concat(
    resultados,
    ignore_index=True
)

# Orden de columnas
rendimientos_mapa_2017_2023 = rendimientos_mapa_2017_2023[
    [
        "ano",
        "cultivo",
        "provincia_mapa",
        "rendimiento_secano_kg_ha",
        "rendimiento_regadio_kg_ha"
    ]
]

print("Dimensiones:", rendimientos_mapa_2017_2023.shape)

print("\nFilas extraídas por cultivo y año:")
display(
    rendimientos_mapa_2017_2023
    .groupby(["cultivo", "ano"])
    .size()
    .unstack()
)

print("\nPrimeras filas:")
display(rendimientos_mapa_2017_2023.head(20))

In [ ]:
# ============================================================
# NORMALIZAR PROVINCIAS MAPA 2017-2023
# Mantiene intacta la tabla original
# ============================================================

rendimientos_mapa_long = rendimientos_mapa_2017_2023.copy()

# 1. Limpiar espacios
rendimientos_mapa_long["provincia_estandar"] = (
    rendimientos_mapa_long["provincia_mapa"]
    .astype("string")
    .str.strip()
)

# 2. Equivalencias MAPA -> nomenclatura estándar
equivalencias_provincias_long = {
    "P. DE ASTURIAS": "Asturias",
    "R. DE MURCIA": "Murcia",
    "S.C. de Tenerife": "Santa Cruz de Tenerife",

    "Vizcaya": "Bizkaia",
    "Guipúzcoa": "Gipuzkoa",
    "Alava": "Álava",

    "CANTABRIA": "Cantabria",
    "BALEARES": "Illes Balears",
    "LA RIOJA": "La Rioja",
    "MADRID": "Madrid",
    "NAVARRA": "Navarra"
}

rendimientos_mapa_long["provincia_estandar"] = (
    rendimientos_mapa_long["provincia_estandar"]
    .replace(equivalencias_provincias_long)
)

# 3. Crear tabla derivada únicamente con las 50 provincias
rendimientos_mapa_provincias = rendimientos_mapa_long[
    rendimientos_mapa_long["provincia_estandar"].isin(provincias_50)
].copy()

# 4. Comprobaciones
print("Filas originales:", len(rendimientos_mapa_2017_2023))
print("Filas provinciales:", len(rendimientos_mapa_provincias))

print("\nNúmero de provincias por cultivo y año:")
display(
    rendimientos_mapa_provincias
    .groupby(["cultivo", "ano"])["provincia_estandar"]
    .nunique()
    .unstack()
)

print("\nDuplicados cultivo × año × provincia:")
print(
    rendimientos_mapa_provincias
    .duplicated(
        subset=["cultivo", "ano", "provincia_estandar"]
    )
    .sum()
)

In [ ]:
# ============================================================
# IDENTIFICAR LA ÚNICA PROVINCIA QUE FALTA EN 2017-2022
# ============================================================

for ano in range(2017, 2023):

    presentes = set(
        rendimientos_mapa_provincias.loc[
            rendimientos_mapa_provincias["ano"] == ano,
            "provincia_estandar"
        ].dropna()
    )

    faltan = sorted(set(provincias_50) - presentes)

    print(f"{ano}: {faltan}")

In [ ]:
# Añadir equivalencia que faltaba
equivalencias_provincias_long["Avila"] = "Ávila"

# Volver a normalizar desde la tabla original
rendimientos_mapa_long = rendimientos_mapa_2017_2023.copy()

rendimientos_mapa_long["provincia_estandar"] = (
    rendimientos_mapa_long["provincia_mapa"]
    .astype("string")
    .str.strip()
    .replace(equivalencias_provincias_long)
)

# Quedarnos solo con las 50 provincias
rendimientos_mapa_provincias = rendimientos_mapa_long[
    rendimientos_mapa_long["provincia_estandar"].isin(provincias_50)
].copy()

# Comprobación final
print("Filas provinciales:", len(rendimientos_mapa_provincias))

display(
    rendimientos_mapa_provincias
    .groupby(["cultivo", "ano"])["provincia_estandar"]
    .nunique()
    .unstack()
)

print(
    "Duplicados:",
    rendimientos_mapa_provincias.duplicated(
        ["cultivo", "ano", "provincia_estandar"]
    ).sum()
)

In [ ]:
# ============================================================
# DISPONIBILIDAD DEL TARGET DE RENDIMIENTO 2017-2023
# ============================================================

control_target = []

for (cultivo, ano), grupo in rendimientos_mapa_provincias.groupby(
    ["cultivo", "ano"]
):

    secano = grupo["rendimiento_secano_kg_ha"]
    regadio = grupo["rendimiento_regadio_kg_ha"]

    control_target.append({
        "cultivo": cultivo,
        "ano": ano,
        "n_provincias": len(grupo),

        "secano_con_dato": secano.notna().sum(),
        "secano_positivo": (secano > 0).sum(),
        "secano_cero": (secano == 0).sum(),

        "regadio_con_dato": regadio.notna().sum(),
        "regadio_positivo": (regadio > 0).sum(),
        "regadio_cero": (regadio == 0).sum(),

        "algun_rendimiento_positivo":
            ((secano > 0) | (regadio > 0)).sum()
    })

control_target_df = pd.DataFrame(control_target)

display(control_target_df)

print("\nTOTAL 2017-2023:")
display(
    control_target_df
    .groupby("cultivo")[
        [
            "n_provincias",
            "secano_positivo",
            "regadio_positivo",
            "algun_rendimiento_positivo"
        ]
    ]
    .sum()
)

#Target Principal del TFM

In [ ]:
# ============================================================
# TARGET PRINCIPAL DEL TFM
# Rendimiento de secano - trigo y cebada - 2017-2023
# ============================================================

target_rendimiento = (
    rendimientos_mapa_provincias[
        rendimientos_mapa_provincias["rendimiento_secano_kg_ha"] > 0
    ]
    [
        [
            "ano",
            "cultivo",
            "provincia_estandar",
            "rendimiento_secano_kg_ha"
        ]
    ]
    .copy()
)

target_rendimiento = target_rendimiento.rename(
    columns={
        "rendimiento_secano_kg_ha": "rendimiento_kg_ha"
    }
)

target_rendimiento = target_rendimiento.sort_values(
    ["cultivo", "ano", "provincia_estandar"]
).reset_index(drop=True)

print("Observaciones totales:", len(target_rendimiento))

print("\nObservaciones por cultivo:")
display(
    target_rendimiento.groupby("cultivo").size()
)

print("\nObservaciones por cultivo y año:")
display(
    target_rendimiento
    .groupby(["cultivo", "ano"])
    .size()
    .unstack()
)

print("\nPrimeras filas:")
display(target_rendimiento.head(10))

# GUARDAR TABLAS

In [ ]:
#-----------------------------
# Comprobar carpeta de salida
#------------------------------

import os

os.makedirs(RUTA_PROCESADOS, exist_ok=True)

print("Carpeta de salida:")
print(RUTA_PROCESADOS)

print("Existe:", os.path.exists(RUTA_PROCESADOS))

In [ ]:
#-------------------------------------
# Guardar el Target definitivo
#--------------------------------------


ruta_target = os.path.join(
    RUTA_PROCESADOS,
    "target_trigo_cebada_2017_2023.csv"
)

target_rendimiento.to_csv(
    ruta_target,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_target)

print("Dimensiones:", target_rendimiento.shape)
print("Columnas:", target_rendimiento.columns.tolist())

In [ ]:
#--------------------------------
# Guardar la tabla MAPA longitudinal completa (Esta la conservaría porque contiene las 700 combinaciones posibles,
#incluyendo secano y regadío. Si más adelante decidimos revisar el target o incorporar regadío, no tendremos que volver a leer los 14 Excel.)
#------------------------------------------

ruta_mapa_long = os.path.join(
    RUTA_PROCESADOS,
    "rendimientos_MAPA_trigo_cebada_2017_2023.csv"
)

rendimientos_mapa_provincias.to_csv(
    ruta_mapa_long,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_mapa_long)

print("Dimensiones:", rendimientos_mapa_provincias.shape)

In [ ]:
#-------------------------------------
# Guardar la tabla que justifica la selección de cultivos
#-------------------------------------

comparativa_final_candidatos

ruta_candidatos = os.path.join(
    RUTA_PROCESADOS,
    "seleccion_cultivos_candidatos.csv"
)

comparativa_final_candidatos.to_csv(
    ruta_candidatos,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_candidatos)

display(comparativa_final_candidatos)

In [ ]:
#---------------------------
# Resumen EuroCrops de trigo y cebada
#---------------------------

cultivos_finales = [
    "common_soft_wheat",
    "barley"
]

eurocrops_final = cultivos_por_provincia_audit[
    cultivos_por_provincia_audit["cultivo_resumen"].isin(cultivos_finales)
].copy()

ruta_eurocrops_final = os.path.join(
    RUTA_PROCESADOS,
    "EuroCrops_trigo_cebada_por_provincia.csv"
)

eurocrops_final.to_csv(
    ruta_eurocrops_final,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_eurocrops_final)

print("Dimensiones:", eurocrops_final.shape)

display(eurocrops_final.head())

In [ ]:
#-------------------------------------
#comprobar que los cuatro archivos existen realmente en Drive
#--------------------------------------

archivos_finales_01 = [
    ruta_target,
    ruta_mapa_long,
    ruta_candidatos,
    ruta_eurocrops_final
]

print("ARCHIVOS FINALES DEL CUADERNO 01\n")

for ruta in archivos_finales_01:
    print(
        os.path.basename(ruta),
        "->",
        "OK" if os.path.exists(ruta) else "NO ENCONTRADO"
    )